In [4]:
df_new_sales = spark.read.option("header",True).csv("Files/raw/sales/sales_2025.csv")

StatementMeta(, cd108f7f-9d01-4a12-8a25-8464506c00fe, 6, Finished, Available, Finished, False)

In [5]:
from pyspark.sql.functions import col

df_new_sales = df_new_sales \
.withColumn("OrderDate", col("OrderDate").cast("date")) \
.withColumn("Quantity", col("Quantity").cast("int")) \
.withColumn("UnitPrice", col("UnitPrice").cast("double")) \
.withColumn("TotalAmount", col("TotalAmount").cast("double"))

StatementMeta(, cd108f7f-9d01-4a12-8a25-8464506c00fe, 7, Finished, Available, Finished, False)

In [6]:
df_new_sales.write.mode("append").saveAsTable("silver_sales")

StatementMeta(, cd108f7f-9d01-4a12-8a25-8464506c00fe, 8, Finished, Available, Finished, False)

In [7]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("OrderID").orderBy(col("OrderDate").desc())

df_dedup = spark.table("silver_sales") \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")


# ================================
# STEP 5 — Clean Gold (One-time safety)
# ================================
dup_count = spark.sql("""
SELECT COUNT(*) as cnt FROM (
    SELECT OrderID
    FROM gold_fact_sales
    GROUP BY OrderID
    HAVING COUNT(*) > 1
) t
""").collect()[0]["cnt"]

if dup_count > 0:
    print(f"Duplicates found in gold: {dup_count} → cleaning...")

    df_gold_clean = spark.table("gold_fact_sales") \
        .withColumn("rn", row_number().over(window)) \
        .filter(col("rn") == 1) \
        .drop("rn")

    df_gold_clean.write.mode("overwrite").saveAsTable("gold_fact_sales")

else:
    print("No duplicates in gold — skipping cleanup")

# ================================
# STEP 6 — MERGE (Best Practice using DeltaTable)
# ================================
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "gold_fact_sales")

delta_table.alias("target").merge(
    df_dedup.alias("source"),
    "target.OrderID = source.OrderID"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()


# ================================
# STEP 7 — Validation (Optional)
# ================================
spark.sql("""
SELECT OrderID, COUNT(*) 
FROM gold_fact_sales
GROUP BY OrderID
HAVING COUNT(*) > 1
""").show()

StatementMeta(, cd108f7f-9d01-4a12-8a25-8464506c00fe, 9, Finished, Available, Finished, False)

No duplicates in gold — skipping cleanup
+-------+--------+
|OrderID|count(1)|
+-------+--------+
+-------+--------+

